# Notebook 8: Calculations and Texts for Streamlit 

In [1]:
#load libraries 
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
import numpy as np
import pandas as pd

In [2]:
# load dataset
df = pd.read_csv("../../data/final_datasets/df_clusters_milieuschutz.csv")
feat = [c for c in df.columns if c.startswith(("soc_", "re_", "com_"))]
Xs = StandardScaler().fit_transform(df[feat].fillna(df[feat].median()))

#index for both variables 
des_idx   = df.index[df["ms_over50"] == 1]
watch_idx = df.index[df["on_watchlist"]]

# Calculate euclidean distance over all 42 standardized dimension features (unweighted)
D = cdist(Xs[watch_idx], Xs[des_idx])
nearest_idx = des_idx[D.argmin(axis=1)]                     

df.loc[watch_idx, "nearest_designated"]     = df.loc[nearest_idx, "plr_name"].values
df.loc[watch_idx, "nearest_designated_bez"] = df.loc[nearest_idx, "bez"].values

watch = df.loc[watch_idx].copy()
watch["rank"] = watch["oof_prob"].rank(ascending=False, method="min").astype(int)

out = watch.sort_values("rank")[
    ["rank", "oof_prob", "plr_id", "plr_name", "bez",
     "nearest_designated", "nearest_designated_bez"]
]
out.to_csv("watchlist_display.csv", index=False)
print(out.to_string(index=False))

 rank  oof_prob   plr_id               plr_name                             bez     nearest_designated          nearest_designated_bez
    1  0.999109  1401044             Uferstraße                      01 - Mitte Westliche Müllerstraße                      01 - Mitte
    2  0.997601  1300731          Koloniestraße                      01 - Mitte           Leopoldplatz                      01 - Mitte
    3  0.995976  1100416            Arkonaplatz                      01 - Mitte      Teutoburger Platz                     03 - Pankow
    4  0.992505  7100206     Großgörschenstraße       07 - Tempelhof-Schöneberg  Richard-Sorge-Viertel   02 - Friedrichshain-Kreuzberg
    5  0.986148  1200623            Stephankiez                      01 - Mitte  Richard-Sorge-Viertel   02 - Friedrichshain-Kreuzberg
    6  0.985959  1401049            Schulstraße                      01 - Mitte     Drontheimer Straße                      01 - Mitte
    7  0.984600  4501147      Leon-Jessel-Platz 04 - Ch

## Nearest Designated Match Interpretation 

The nearest designated match for each watchlist PLR often lies in a different part of the city (Großgörschenstraße in Schöneberg maps to Richard-Sorge-Viertel in Friedrichshain, for example), which is meaningful because geography is not a model feature. The resemblance is structural, not spatial. 

Several designated areas recur as the closest match for multiple watchlist PLR (Richard-Wagner-Straße three times, Teutoburger Platz and Letteplatz twice each), reflecting that different undesignated areas can share the same designated profile. The four Charlottenburg-Wilmersdorf entries (Leon-Jessel-Platz, Schloßstraße, Amtsgerichtsplatz, Wilhelmsaue) cluster in the lower half of the ranking, consistent with the weaker signal in that region already documented as a data property, not a model limitation. 

The comparison rests on the same features as the resemblance score and uses unweighted Euclidean distance, so it should be read as an illustrative reference area rather than an independent confirmation.

## Texts for Streamlit Profiles 

Undesignated 

Similarity score {score} 

- **Not majority-protected**: less than half of this planning area lies within a designated Milieuschutz area (§172), so it is treated here as undesignated.
- **Not on the watchlist**: this area is not among the 25 planning areas whose profile most closely matches Berlin's designated Milieuschutz neighbourhoods.
- **Similarity score {score}**: how closely this area's overall profile matches the pattern of the already-designated neighbourhoods — higher means more similar. It reflects how the area looks today, not a guarantee about the future.
- **Resemblance, not risk**: this is a similarity to the designated pattern, not a forecast of gentrification or displacement, and it says nothing about individual residents.
- **Not on the watchlist ≠ safe**: the underlying label reflects political designation, not confirmed displacement, and areas change over time; not appearing today means "does not currently resemble the designated areas", not "will never qualify".

Designated 
-	**Under Milieuschutz protection**: more than half of this planning area lies within a designated Milieuschutz area (§172 social-preservation status) under Berlin housing policy.
-	**Part of the reference set**: designated areas are the examples the tool learns from — their shared profile defines the pattern that the watchlist looks for elsewhere. 
-	**Reference, not prediction**: because this area is already protected, it is part of what the tool learns from rather than something it scores; no similarity score is shown for it. 
-	**Designation is political, not automatic**: The status reflects an administrative district-level decision to protect the area, not a measured level of gentrification or displacement — the tool treats it as a given, not as something it evaluates.

Watchlist 

On the watchlist · rank {rank} of 25
Similarity score {score} 
Closest designated match: {nearest_designated}

-	**On the watchlist**: this undesignated area is among the 25 closest matches to Berlin's already-designated Milieuschutz neighbourhoods.
- **Similarity score {score}**: how closely this area's overall profile matches the pattern of the already-designated neighbourhoods — higher means more similar. It reflects how the area looks today, not a forecast of gentrification or displacement.	
-   **Closest designated match**: the already-protected neighbourhood whose overall profile most resembles this area. It is often in a different part of the city, since the match is based on structural similarity of social, real-estate and commercial patterns. Read it as an illustrative reference point ("this area most resembles that protected one"), not as proof that the two are alike in every respect.
-	**A flag for closer review, not a verdict**: A high match means the area may warrant a closer look by the district — a starting point for a preliminary study (vorbereitende Untersuchung), not a sign that anything is already happening.
-	**Compared against a political label**: the designated areas the score matches against reflect administrative decisions, not confirmed displacement; the watchlist surfaces where that same pattern appears in areas not yet protected.



# **KiezKeeper - an early-warning map for neighbourhood protection in Berlin**

## Introduction page 

**The problem**: In Berlin, tenant protection against displacement (known as Milieuschutz, §172) is patchwork, depending both on political will and a long administrative process, and therefore usually arrives late. By the time the process recognises that a neighbourhood is changing, much of that change has already happened. And because each of the city's twelve districts decides on its own, there has never been one shared, city-wide picture of which areas are under pressure.

**KiezKeeper turns this around**: instead of reacting after the fact, it looks across all of Berlin at once. It learns what the city's already-protected neighbourhoods have in common on three dimensions: their social make-up, their housing market, and their local business landscape. From these patterns, KiezKeeper does two things: it sorts every neighbourhood into a profile that captures what kind of area it is, and it flags the unprotected ones whose profile most closely resembles the areas already under protection. 

Together this gives Berlin a single, consistent view: turning twelve separate district perspectives into one shared, city-wide map.

**What it is — and what it isn't**: KiezKeeper measures resemblance, not risk. An area on the watchlist looks like the neighbourhoods a Berlin district has already chosen to protect — that is a reason to take a closer look, not proof that displacement is happening, and not a judgement about the people who live there. The aim is not to measure change once it is over. It is to give the whole city a way to act while there is still a neighbourhood to keep.

## Page with Plots 

### Description Cluster 

KiezKeeper groups all of Berlin's neighbourhoods into four profiles based on the three dimensions, i.e. areas that share a similar structural make-up. These profiles are found in the data itself, not defined in advance, and can be read as different stages along the path of neighbourhood change.

### Description Watchlist  

Within the unprotected neighbourhoods, KiezKeeper then compares each area against the ones Berlin has already chosen to protect and measures how closely their profiles match. The areas that most closely resemble the designated Milieuschutz pattern form the watchlist: a Berlin-wide, objective and consistent starting point for asking where protection might be needed next.


